# Compact Adaptive QIR profiling throughput

Measure `SequentialCountingProfiler` without interpreting Q# at runtime. The application executes **100,000,000 loop gates** on one qubit, plus four setup/cleanup instructions with the LLVM JIT (three with bytecode).

A measurement determines both the initial loop counter and its stopping point. This prevents compile-time loop expansion while keeping the number of iterations identical for either measurement outcome. The bounded loop body contains 100 alternating H/T gates to amortize loop-control overhead. There are no measurements inside the loop, so gate batches are not interrupted by result barriers.

Two measurements distinguish costs:
1. **End-to-end `use_qir=True`:** Q# compilation, QIR parsing, LLVM JIT compilation, and native execution. This restores the inkwell runner removed in commit `00bd4e0ac`.
2. **Pre-lowered bytecode:** reuse the same Adaptive bytecode and time its execution through a fresh counting profiler.

Both report wall-clock seconds and ns per counted backend instruction, not simulated quantum execution time. JIT execution emits `mz_store_reset` followed by `read_result`; bytecode uses one immediate `mz_reset`. Counts are validated separately without a result adapter, so both paths measure the counting backend directly. **20 ns/op is a reference, not an asserted result:** at this scale it corresponds to about two seconds per run.

In [1]:
from statistics import median
from time import perf_counter

import pyqir
from qdk import Context, TargetProfile
from qdk._adaptive_pass import AdaptiveProfilePass, Bytecode
from qdk.qre import SequentialCountingProfiler
from qdk.qre.application import QSharpApplication
from qdk.qre.profiler import QREProfiler

TARGET_GATES = 100_000_000
GATES_PER_ITERATION = 100
REPEATS = 5

assert TARGET_GATES > 0 and GATES_PER_ITERATION > 0
assert GATES_PER_ITERATION % 2 == 0
assert TARGET_GATES % GATES_PER_ITERATION == 0
assert REPEATS > 0
ITERATIONS = TARGET_GATES // GATES_PER_ITERATION

context = Context(target_profile=TargetProfile.Adaptive)
# Expand only the bounded gate block, never the million-iteration outer loop.
gate_block = "\n".join(["H(q); T(q);"] * (GATES_PER_ITERATION // 2))
qsharp_source = f"""
/// Execute a fixed gate block in a measurement-dependent runtime loop.
operation CompactLoop(iterations : Int) : Unit {{
    use q = Qubit();
    H(q);
    let start = if MResetZ(q) == One {{ 1 }} else {{ 0 }};
    mutable i = start;
    let stop = start + iterations;
    while i < stop {{
        {gate_block}
        set i += 1;
    }}
    Reset(q);
}}
"""
context.eval(qsharp_source)
app = QSharpApplication(entry_expr=context.code.CompactLoop, args=(ITERATIONS,))
print(f"{ITERATIONS:,} iterations, {TARGET_GATES:,} loop gates, one qubit")

1,000,000 iterations, 100,000,000 loop gates, one qubit


In [2]:
def make_profiler():
    """Count the emitted instructions without assigning hardware durations."""
    return QREProfiler(
        SequentialCountingProfiler(
            physical_qubits_per_logical=1,
            gate_time_sec={
                name: 0.0
                for name in ("h", "t", "mz_reset", "mz_store_reset", "read_result", "reset")
            },
        )
    )


def validate_result(result, iterations, *, use_jit=True):
    """Verify every gate was executed and the operation denominator is exact."""
    pairs = iterations * (GATES_PER_ITERATION // 2)
    expected = {"h": pairs + 1, "t": pairs, "reset": 1}
    if use_jit:
        expected.update({"mz_store_reset": 1, "read_result": 1})
    else:
        expected["mz_reset"] = 1
    assert result.totals.calls_per_operation == expected, result
    assert result.totals.total_operations_count == sum(expected.values())
    assert result.totals.logical_qubits == result.totals.physical_qubits == 1
    assert not result.infos, result.infos
    return result.totals.total_operations_count


def benchmark(label, profile_once, *, use_jit=True):
    """Time repeated profiles with fresh counters and validate outside the timer."""
    elapsed_samples = []
    for repeat in range(REPEATS):
        profiler = make_profiler()
        t0 = perf_counter()
        result = profile_once(profiler)
        elapsed = perf_counter() - t0
        count = validate_result(result, ITERATIONS, use_jit=use_jit)
        elapsed_samples.append(elapsed)
        print(
            f"{label}, run {repeat + 1}: {count:,} ops, "
            f"{elapsed:.4f}s, {1e9 * elapsed / count:.2f} ns/op",
            flush=True,
        )
    return {
        "label": label,
        "operations": count,
        "median_sec": median(elapsed_samples),
        "min_sec": min(elapsed_samples),
        "max_sec": max(elapsed_samples),
        "median_ns_per_op": 1e9 * median(elapsed_samples) / count,
    }

## Verify that the compiler retained the loop

Compile both 10 iterations and the full workload. Require matching function/block/instruction counts, genuine backward branches, and QIR smaller than 20 KB. Only the trip-count constant should grow with the workload. These checks fail rather than accidentally benchmarking a gigantic unrolled program.

In [3]:
def inspect_qir(text):
    """Measure QIR size and find branches to earlier blocks within each function."""
    module = pyqir.Module.from_ir(pyqir.Context(), text)
    error = module.verify()
    if error is not None:
        raise ValueError(error)
    functions = [function for function in module.functions if function.basic_blocks]
    back_edges = []
    block_count = instruction_count = 0
    for function in functions:
        blocks = function.basic_blocks
        positions = {block.name: index for index, block in enumerate(blocks)}
        block_count += len(blocks)
        for index, block in enumerate(blocks):
            instruction_count += len(block.instructions)
            terminator = block.instructions[-1]
            if terminator.opcode == pyqir.Opcode.BR:
                for operand in terminator.operands:
                    if isinstance(operand, pyqir.BasicBlock) and positions[operand.name] <= index:
                        back_edges.append((function.name, block.name, operand.name))
    return {
        "bytes": len(text.encode("utf-8")),
        "functions": len(functions),
        "blocks": block_count,
        "instructions": instruction_count,
        "back_edges": back_edges,
    }


small_qir = str(context.compile(context.code.CompactLoop, 10))
t0 = perf_counter()
qir_text = str(context.compile(context.code.CompactLoop, ITERATIONS))
compile_sec = perf_counter() - t0
small_shape = inspect_qir(small_qir)
large_shape = inspect_qir(qir_text)
assert '"qir_profiles"="adaptive_profile"' in qir_text
assert large_shape["back_edges"], "No runtime loop found"
for key in ("functions", "blocks", "instructions", "back_edges"):
    assert small_shape[key] == large_shape[key], (small_shape, large_shape)
assert large_shape["bytes"] < 20_000, large_shape
assert abs(large_shape["bytes"] - small_shape["bytes"]) < 64
print(f"QIR size: {small_shape['bytes']:,} -> {large_shape['bytes']:,} bytes")
print(f"Q# -> QIR compilation: {compile_sec:.4f}s")
print("Full-workload QIR structure:", large_shape)

small_app = QSharpApplication(entry_expr=context.code.CompactLoop, args=(10,))
warmup = make_profiler().profile(small_app, use_qir=True)
validate_result(warmup, 10)
print("Small-run gate counts verified:", warmup.totals.calls_per_operation)

QIR size: 7,324 -> 7,329 bytes
Q# -> QIR compilation: 0.0545s
Full-workload QIR structure: {'bytes': 7329, 'functions': 4, 'blocks': 10, 'instructions': 143, 'back_edges': [('ENTRYPOINT__main', 'block_5', 'block_4')]}
Small-run gate counts verified: {'t': 500, 'h': 501, 'mz_store_reset': 1, 'read_result': 1, 'reset': 1}


## End-to-end profiling with `use_qir=True`

This is the public notebook API. Each timed call compiles Q# to Adaptive QIR, parses/lowers it to bytecode, and executes it through the counting profiler. Profiler construction and result validation are outside the timer.

In [4]:
end_to_end = benchmark(
    "use_qir=True (LLVM JIT)",
    lambda profiler: profiler.profile(app, use_qir=True),
)

use_qir=True (LLVM JIT), run 1: 100,000,004 ops, 0.9068s, 9.07 ns/op


use_qir=True (LLVM JIT), run 2: 100,000,004 ops, 0.9058s, 9.06 ns/op


use_qir=True (LLVM JIT), run 3: 100,000,004 ops, 0.9177s, 9.18 ns/op


use_qir=True (LLVM JIT), run 4: 100,000,004 ops, 0.9030s, 9.03 ns/op


use_qir=True (LLVM JIT), run 5: 100,000,004 ops, 0.9711s, 9.71 ns/op


## Isolate pre-lowered bytecode execution

Keep the former bytecode path as a comparison: lower with `AdaptiveProfilePass(Bytecode.Bit64)` once outside the execution timer. This diagnostic uses an internal lowering API; the preceding benchmark uses the public `use_qir=True` path, which now invokes the LLVM JIT instead.

The execution timer still includes the small Python/Rust bytecode-deserialization and result-conversion costs. This is a gate-heavy, measurement-light throughput benchmark, not a prediction for every adaptive application.

In [5]:
t0 = perf_counter()
qir_module = pyqir.Module.from_ir(pyqir.Context(), qir_text)
program = AdaptiveProfilePass(Bytecode.Bit64).run(qir_module)
program_dict = program.as_dict()
lowering_sec = perf_counter() - t0
assert len(program.instructions) < 10_000
print(
    f"QIR parsing + bytecode lowering/serialization: {lowering_sec:.4f}s; "
    f"{len(program.instructions)} static bytecode instructions, "
    f"{len(program.quantum_ops)} static quantum-op records"
)

execution_only = benchmark(
    "Pre-lowered bytecode",
    lambda profiler: profiler.profiler.profile_adaptive_program(program_dict),
    use_jit=False,
)

print("\nSummary (median of", REPEATS, "runs):")
for measurement in (end_to_end, execution_only):
    print(
        f"{measurement['label']}: {measurement['median_sec']:.4f}s, "
        f"{measurement['median_ns_per_op']:.2f} ns/op "
        f"(time range {measurement['min_sec']:.4f}-{measurement['max_sec']:.4f}s)"
    )
print(f"20 ns/op reference: {(TARGET_GATES + 4) * 20e-9:.4f}s")
print(
    f"LLVM JIT speedup over bytecode: "
    f"{execution_only['median_ns_per_op'] / end_to_end['median_ns_per_op']:.2f}x"
)

QIR parsing + bytecode lowering/serialization: 0.0045s; 244 static bytecode instructions, 4 static quantum-op records


Pre-lowered bytecode, run 1: 100,000,003 ops, 4.5787s, 45.79 ns/op


Pre-lowered bytecode, run 2: 100,000,003 ops, 4.4183s, 44.18 ns/op


Pre-lowered bytecode, run 3: 100,000,003 ops, 4.4186s, 44.19 ns/op


Pre-lowered bytecode, run 4: 100,000,003 ops, 4.4325s, 44.33 ns/op


Pre-lowered bytecode, run 5: 100,000,003 ops, 4.6048s, 46.05 ns/op



Summary (median of 5 runs):
use_qir=True (LLVM JIT): 0.9068s, 9.07 ns/op (time range 0.9030-0.9711s)
Pre-lowered bytecode: 4.4325s, 44.33 ns/op (time range 4.4183-4.6048s)
20 ns/op reference: 2.0000s
LLVM JIT speedup over bytecode: 4.89x
